# Weather & Energy Pipeline — Spark Processing (Member 2)

**Course:** GIK2Q3 Applied Big Data and Cloud Computing  
**Team:** B5  
**Member:** 2 — Spark Developer  

---

## Notebook Overview

This notebook contains the **Apache Spark implementation** of the Weather & Energy pipeline. The goal is to transform the Bronze layer data into **Silver** and **Gold layers** suitable for analysis, following the VG (Distinction) project requirements.

We use **PySpark DataFrames** to:

1. **Load Bronze Parquet files**  
   - Weather data from SMHI  
   - Energy prices from Nordpool  

2. **Clean and filter the data (Silver layer)**  
   - Remove invalid or null temperature readings  
   - Convert temperature to numeric  
   - Add timestamp, date, hour, month, and year columns  

3. **Aggregate and enrich the data (Gold layer)**  
   - Compute hourly average, min, and max temperature per price zone  
   - Join with energy prices  
   - Implement **rolling 24-hour temperature averages** using Spark window functions  
   - Rank monthly spot prices for analysis  

4. **Track execution times and log events**  
   - Time taken for each stage (Silver and Gold)  
   - Number of records processed  
   - Log messages for reproducibility and debugging  

---

## Notes

- **Prerequisites:** Bronze layer Parquet files (`weather_raw.parquet` and `energy_raw.parquet`) must exist.  
- **Output:**  
  - Silver layer: `data/silver_spark/weather_clean.parquet`  
  - Gold layer: `data/gold/gold_weather_energy.parquet`  

---

**Next Step:** Start the Spark session and configure logging for the pipeline.

## Install Required Python Packages

Run the following command in your terminal to install all necessary Python libraries for the Weather & Energy pipeline:

```bash
pip install -r C:\Users\USER\weather_energy_pipeline_\requirements.txt

## Verify Environment and Libraries

Before starting the Spark and Dask pipeline, we need to ensure all required tools and libraries are installed and working correctly.

This cell will:

1. **Check Apache Spark** — start a Spark session and print the version.  
2. **Check Dask** — start a Dask client and display the dashboard link.  
3. **Check Pandas and PyArrow** — confirm versions for DataFrame operations and Parquet support.  
4. **Check Plotting Libraries** — Matplotlib and Seaborn for data visualization.

If all checks pass, the pipeline environment is ready for use.

In [1]:
import os
import sys

os.environ["JAVA_HOME"]             = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot"
os.environ["PATH"]                  += os.pathsep + os.path.join(os.environ["JAVA_HOME"], "bin")
os.environ["HADOOP_HOME"]           = r"C:\hadoop"
os.environ["PATH"]                  += os.pathsep + r"C:\hadoop\bin" 
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"]        = "127.0.0.1"

print("Environment configured!")

Environment configured!


In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('test').getOrCreate()
print('Spark OK:', spark.version)

import dask.dataframe as dd
from dask.distributed import Client
client = Client()
print('Dask OK:', client)

import pandas as pd
import pyarrow
print('Pandas OK:', pd.__version__)

import matplotlib.pyplot as plt
import seaborn as sns
print('Plotting OK')


Spark OK: 3.5.0
Dask OK: <Client: 'tcp://127.0.0.1:56317' processes=4 threads=12, memory=15.65 GiB>
Pandas OK: 3.0.1
Plotting OK


## Create Spark Session

### A Spark session is created with the following configuration:

Application Name: WeatherEnergyPipeline-Spark

Driver Memory: 8 GB

Shuffle Partitions: 50 (controls parallelism during data shuffling)

This configuration is suitable for medium-sized datasets and helps optimize distributed data processing.

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import time, logging, shutil, os

logging.basicConfig(level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

spark = SparkSession.builder \
    .appName('WeatherEnergyPipeline-Spark') \
    .config("spark.ui.enabled", "true") \
    .config('spark.driver.memory', '8g') \
    .config("spark.executor.memory", "8g") \
    .config("spark.ui.port", "4040") \
    .config('spark.sql.shuffle.partitions', '50') \
    .getOrCreate()

logger.info(f'Spark version: {spark.version}')

# --- Print Spark UI URL ---
spark_ui_url = spark.sparkContext.uiWebUrl
logger.info(f'Spark UI URL: {spark_ui_url}')

2026-03-21 06:46:44,121 - INFO - Spark version: 3.5.0
2026-03-21 06:46:44,127 - INFO - Spark UI URL: http://100.84.40.19:4040


## Spark Silver Stage: Weather Data Cleaning

This section reads raw weather data from Bronze, filters invalid records, and writes cleaned data to the Silver layer.

---

### Paths

```python
bronze_weather_path = r"C:/Users/USER/weather_energy_pipeline_/data/bronze/weather_raw.parquet"
bronze_energy_path  = r"C:/Users/USER/weather_energy_pipeline_/data/bronze/energy_raw.parquet"
silver_weather_path = r"C:/Users/USER/weather_energy_pipeline_/data/silver_spark/weather_clean"

In [5]:

# ---------------------------
# Paths
bronze_weather_path = r"C:/Users/USER/weather_energy_pipeline_/data/bronze/weather_raw.parquet"
bronze_energy_path  = r"C:/Users/USER/weather_energy_pipeline_/data/bronze/energy_raw.parquet"
silver_weather_path = r"C:/Users/USER/weather_energy_pipeline_/data/silver_spark/weather_clean"

# ---------------------------
# Timer start
spark_total_start = time.time()
silver_start = time.time()

# ---------------------------
# Read Bronze parquet files
weather = spark.read.parquet(bronze_weather_path)
energy  = spark.read.parquet(bronze_energy_path)
logger.info(f'Weather: {weather.count():,} | Energy: {energy.count():,}')
logger.info(f'Raw data loaded in {time.time()-silver_start:.1f}s')

# ---------------------------
# Filter invalid records and remove ancient timestamps
weather_clean = weather \
    .filter(F.col('Lufttemperatur').isNotNull()) \
    .filter(F.col('Lufttemperatur').cast('double').between(-60, 45)) \
    .withColumn('temperature', F.col('Lufttemperatur').cast('double')) \
    .withColumn('timestamp', F.to_timestamp(
        F.concat(F.col('Datum'), F.lit(' '), F.col('Tid (UTC)')),
        'yyyy-MM-dd HH:mm:ss')) \
    .withColumn('date', F.to_date('timestamp'
                                 )) \
    .withColumn('hour', F.hour('timestamp')) \
    .withColumn('month', F.month('timestamp')) \
    .withColumn('year', F.year('timestamp')) \
    .filter(F.col('timestamp') >= F.lit('1900-01-01'))

logger.info(f'Filtering done | Remaining rows: {weather_clean.count():,}')

# ---------------------------
# Repartition to avoid Windows commit issues
weather_clean = weather_clean.repartition(20)
logger.info('Repartition done')

# ---------------------------
# Clean output folder (important on Windows)
if os.path.exists(silver_weather_path):
    shutil.rmtree(silver_weather_path)

# ---------------------------
# Write Silver parquet
weather_clean.write \
    .mode('overwrite') \
    .parquet(silver_weather_path)

silver_time = time.time() - silver_start
logger.info(f'Spark Silver done in {silver_time:.1f}s | Records: {weather_clean.count():,}')

2026-03-21 06:46:58,309 - INFO - Weather: 73,355,669 | Energy: 210,340
2026-03-21 06:46:58,309 - INFO - Raw data loaded in 1.3s
2026-03-21 06:47:25,795 - INFO - Filtering done | Remaining rows: 72,176,442
2026-03-21 06:47:25,810 - INFO - Repartition done
2026-03-21 06:52:11,767 - INFO - Spark Silver done in 273.1s | Records: 72,176,442


## Spark Gold Stage: Weather & Energy Aggregation and Join

This stage reads the Silver weather data and Bronze energy data, aggregates them, joins on common keys, ranks monthly spot prices, and writes the final Gold parquet dataset.


In [6]:

gold_start = time.time()

# =========================================================
# Paths
# =========================================================
gold_output_path    = r"C:/Users/USER/weather_energy_pipeline_/data/gold/gold_weather_energy.parquet"

# =========================================================
# READ SILVER + ENERGY
# =========================================================
weather = spark.read.parquet(silver_weather_path)
energy  = spark.read.parquet(bronze_energy_path)

logger.info(f"Weather (Silver): {weather.count():,}")
logger.info(f"Energy (Bronze): {energy.count():,}")

# =========================================================
# ENERGY CLEANING (ALIGN WITH WEATHER)
# =========================================================
energy_clean = energy \
    .withColumn("timestamp", F.to_timestamp("datetime")) \
    .withColumn("date", F.to_date("timestamp")) \
    .withColumn("hour", F.hour("timestamp")) \
    .withColumn("month", F.month("timestamp")) \
    .withColumn("year", F.year("timestamp")) \
    .withColumnRenamed("spot_price_sek", "spot_price")

# =========================================================
# WEATHER AGGREGATION (FIXED)
# =========================================================
weather_agg = weather.groupBy(
    "price_zone", "date", "hour", "year", "month"
).agg(
    F.avg("temperature").alias("avg_temp"),
    F.min("temperature").alias("min_temp"),
    F.max("temperature").alias("max_temp"),
    F.countDistinct("station_id").alias("station_count")
)

# =========================================================
# ENERGY AGGREGATION
# =========================================================
energy_agg = energy_clean.groupBy(
    "price_zone", "date", "hour", "year", "month"
).agg(
    F.avg("spot_price").alias("avg_spot_price")
)

# =========================================================
# JOIN
# =========================================================
gold = weather_agg.join(
    energy_agg,
    on=["price_zone", "date", "hour", "year", "month"],
    how="inner"
)
print(f"After join: {gold.count():,} records")

# =========================================================
# OPERATION 5: 24H ROLLING AVG TEMP
# =========================================================
window_24h = Window \
    .partitionBy('price_zone') \
    .orderBy(F.col('date').cast('long') * 24 + F.col('hour')) \
    .rowsBetween(-23, 0)

gold = gold.withColumn(
    'rolling_24h_avg_temp',
    F.avg('avg_temp').over(window_24h)
)

# =========================================================
# RANKING
# =========================================================
window_rank = Window.partitionBy("year", "month") \
    .orderBy(F.desc("avg_spot_price"))

gold = gold.withColumn(
    "monthly_price_rank",
    F.rank().over(window_rank)
)

# =========================================================
# WRITE GOLD
# =========================================================
if os.path.exists(gold_output_path):
    shutil.rmtree(gold_output_path)

gold.write \
    .mode("overwrite") \
    .partitionBy("year", "month") \
    .parquet(gold_output_path)

# =========================================================
# LOGGING + DEBUG
# =========================================================
gold_time = time.time() - gold_start
spark_total_time = time.time() - spark_total_start
logger.info(f"Gold completed in {gold_time:.1f}s")
logger.info(f"TOTAL SPARK TIME in {spark_total_time:.1f}s")

print("Gold Schema:")
gold.printSchema()

print("Sample Data:")
gold.show(5)

2026-03-21 06:52:47,009 - INFO - Weather (Silver): 72,176,442
2026-03-21 06:52:47,217 - INFO - Energy (Bronze): 210,340


After join: 210,316 records


2026-03-21 06:56:46,415 - INFO - Gold completed in 241.3s
2026-03-21 06:56:46,518 - INFO - TOTAL SPARK TIME in 589.4s


Gold Schema:
root
 |-- price_zone: string (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- avg_temp: double (nullable = true)
 |-- min_temp: double (nullable = true)
 |-- max_temp: double (nullable = true)
 |-- station_count: long (nullable = false)
 |-- avg_spot_price: double (nullable = true)
 |-- rolling_24h_avg_temp: double (nullable = true)
 |-- monthly_price_rank: integer (nullable = false)

Sample Data:
+----------+----------+----+----+-----+------------------+--------+--------+-------------+--------------+--------------------+------------------+
|price_zone|      date|hour|year|month|          avg_temp|min_temp|max_temp|station_count|avg_spot_price|rolling_24h_avg_temp|monthly_price_rank|
+----------+----------+----+----+-----+------------------+--------+--------+-------------+--------------+--------------------+------------------+
|       SE4|2019-08-20|   7|